# Subcellular fractions vs Total: fracSanity, cleavage-site usage and APA (siCtrl)

HCT116 cells: RNA of FACS-sorted subcellular fractions and Total RNA, sequenced with ONT (`nanoflowz` pipeline), **siCtrl libraries only**. This notebook follows `pbody_RNA_Spang.fracSanity.pcTPM.ipynb` (P-body pull-down vs Total), with the sorted fraction in the place of the pull-down (PD), and uses [Sanity_utils](https://github.com/nimwegenLab/Sanity_utils) throughout. Read QC, nuclear contamination and the compression of the fractions are in `ONT_organelles_vs_Total.ipynb`.

**Quantities.**
- $\alpha_g$: fraction of the mRNA molecules of gene $g$ in the cell that end up in the sorted fraction (fracSanity, [docs](https://github.com/nimwegenLab/Sanity_utils/blob/main/docs/fracSanity.md)). The libraries were made from the sorted material, not from a fixed share of the cells, so $\alpha_g$ is the fraction **up to the factor $F$**, see below.
- $F$: fraction of all poly(A) RNA molecules of the cell in the sorted fraction (abundance-weighted mean of the $\alpha_g$). Without spike-ins or a measured yield, $F$ is weakly determined (as in the P-body analysis); there is no imaging calibration here, so $F$ has a uniform prior. **What the data determine well is the ranking of genes within a fraction**: $\log_2\alpha_g - \log_2 F = D_g$ = log2 CPM in the fraction minus log2 CPM in Total.
- Over- and under-representation of functional gene sets in each fraction: ISMARA-style activities ([docs](https://github.com/nimwegenLab/Sanity_utils/blob/main/docs/gene_set_analysis.md)) on the per-library $\log_2\alpha_g$, centred per library, i.e. relative to the average gene of the fraction.
- Cleavage sites (CS), every read 3' end position its own CS: relative usage of each CS among the CS of its gene ([isoform analysis](https://github.com/nimwegenLab/Sanity_utils/blob/main/docs/isoform_analysis.md)), 3' UTR shortening / lengthening (tandem CS in the terminal exon of the gene) in each fraction relative to Total ([APA analysis](https://github.com/nimwegenLab/Sanity_utils/blob/main/docs/apa_analysis.md)), and $\alpha$ of every CS (fracSanity on the CS counts).

**Design choices.**
- **siCtrl only.** siCPSF5 changes both polyadenylation and, indirectly, expression; differential recruitment siCtrl vs siCPSF5 is not done here.
- **Gene universe: all genes of the annotation** (GENCODE v42 basic) with at least one read, including non-coding and mitochondrially encoded genes, without length correction: ONT reads are full-length molecules, one read per molecule, so Sanity's log2 expression is log2 CPM.
- **Replicates are not paired.** Fraction libraries and Total libraries come from different experiments: every fraction (cohort average, or single library) is compared with the average of the three siCtrl Total libraries.
- **Cohort = batch and fraction** (e.g. `Golgi/PEX: Free`): Free and Org2 were prepared in each of the three batches and are kept apart.
- Sanity with `v_method='EAP'` (the docs recommend it over MAP for features with few counts, which applies to the shallow fraction libraries and to cleavage sites); ISMARA replicate averages with `method='reml'`; fracSanity with the outlier fraction bounded by $\pi \le 0.5$, as in the P-body analysis.

The heavy steps run in `WF/scripts/ONT_fracSanity_pipeline.py` (SLURM job `ONT_fracSanity.sbatch`); this notebook reads their outputs and fits the gene-set activities.

**Kernel.** `sanity_utils` conda environment.

In [ ]:
import warnings
warnings.simplefilter('ignore')

import os
import re
import csv
import glob
import pickle

import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.stats import spearmanr

from dotenv import load_dotenv

import seaborn as sns; sns.set()
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, HTML
%matplotlib inline

import sanity_utils as su
from sanity_utils import plotting as su_plotting
from sanity_utils.genesets import (GeneSetCollection, set_source_table, load_go, load_reactome, load_kegg,
                                   load_wikipathways, read_ncbi_gene_info, latest_release)
print("sanity_utils", su.__version__, "| Sanity", su.get_sanity_version(), su.get_sanity_commit())

In [ ]:
load_dotenv("APA_localization.scicore.env", override=True)
subdirs = {"wf_runs_dir": os.getenv("WF_RUNS_DIR"), "figures_dir": os.getenv("FIGURES_DIR"), "tables_dir": os.getenv("TABLES_DIR"),
           "external_data_dir": os.getenv("EXTERNAL_DATA_DIR")}

ANALYSIS = "ONT_organelles_fracSanity"
CS_LEVEL = "cs_pos"                      # one CS per read 3' end position (pipeline level name)
SU_OUT = os.path.join(subdirs["wf_runs_dir"], "Sanity_utils_ONT_organelles_siCtrl")   # WF/scripts/ONT_fracSanity_pipeline.py
GENESETS_DIR = os.path.join(subdirs["external_data_dir"], "genesets_human")           # sanity-utils-fetch-genesets all --organism human
if os.environ.get("ONT_FRACSANITY_TEST_DIR"):
    # test runs of the notebook on a reduced data set: inputs and outputs in a separate folder
    _test = os.environ["ONT_FRACSANITY_TEST_DIR"]
    SU_OUT, subdirs["figures_dir"], subdirs["tables_dir"] = (os.path.join(_test, d) for d in ("out", "figures", "tables"))
FIGS = os.path.join(subdirs["figures_dir"], "gene_expression_analysis", ANALYSIS)
TABLES = os.path.join(subdirs["tables_dir"], "gene_expression_analysis", ANALYSIS)

def figure_caption(path, formats=("png", "pdf")):
    # file name of a saved figure, shown with the plot; the links point to the full-size files relative to the
    # HTML report (WF/scripts/make_html_reports.py copies them to <report>/figures/, same subfolders as FIGURES_DIR)
    path = os.path.splitext(path)[0] if path.endswith((".png", ".pdf")) else path
    rel = os.path.relpath(path, subdirs["figures_dir"])
    links = " | ".join(f'<a href="figures/{rel}.{f}" target="_blank">{f}</a>' for f in formats)
    display(HTML(f'<p><b>Figure:</b> <code>{os.path.basename(path)}</code> &nbsp;({links})</p>'))

def savefig(fig, subdir, name, formats=("png", "pdf"), dpi=300):
    out = os.path.join(FIGS, subdir)
    os.makedirs(out, exist_ok=True)
    for fmt in formats:
        fig.savefig(os.path.join(out, f"{name}.{fmt}"), bbox_inches="tight", dpi=dpi)
    figure_caption(os.path.join(out, name), formats)

def save_tsv(df, subdir, name, index=False):
    out = os.path.join(TABLES, subdir)
    os.makedirs(out, exist_ok=True)
    df.to_csv(os.path.join(out, name), sep="\t", header=True, index=index, quoting=csv.QUOTE_NONE)
    return os.path.join(out, name)

def load_pickle(path):
    with open(path, "rb") as fh:
        return pickle.load(fh)

def safe(name):
    # file name of a cohort or library (as in the pipeline)
    return re.sub(r"[^A-Za-z0-9]+", "_", name)

# notation as in ONT_organelles_vs_Total.ipynb: color = fraction; marker = batch (all libraries here are siCtrl)
FRACTION_ORDER = ["Total", "Free", "Org2", "ER", "TIS", "Golgi", "PEX", "Endo", "TJ"]
FRACTION_COLORS = {"Total": "black", "Free": "tab:blue", "Org2": "tab:orange", "ER": "tab:green", "TIS": "tab:red", "Golgi": "tab:purple",
                   "PEX": "tab:brown", "Endo": "tab:pink", "TJ": "tab:olive"}
BATCH_ORDER = ["HCT116 Total", "ER/TIS", "Golgi/PEX", "End/TJ"]
BATCH_MARKERS = {"HCT116 Total": "D", "ER/TIS": "o", "Golgi/PEX": "s", "End/TJ": "^"}
BATCH_COLORS = {"HCT116 Total": "black", "ER/TIS": "0.75", "Golgi/PEX": "0.5", "End/TJ": "0.25"}

def notation_handles(fractions, batches):
    return ([matplotlib.lines.Line2D([], [], color=FRACTION_COLORS[f], marker="o", ls="", label=f) for f in FRACTION_ORDER if f in set(fractions)]
            + [matplotlib.lines.Line2D([], [], color="grey", marker=BATCH_MARKERS[b], ls="", label=b) for b in BATCH_ORDER if b in set(batches)])

def color_ticklabels(labels, fractions):
    for t, f in zip(labels, fractions):
        t.set_color(FRACTION_COLORS.get(f, "black"))
os.makedirs(FIGS, exist_ok=True); os.makedirs(TABLES, exist_ok=True)
print(SU_OUT)

## Pipeline

`WF/scripts/ONT_fracSanity_pipeline.py`, run by `sbatch ONT_fracSanity.sbatch` (steps whose outputs exist are skipped, nothing is overwritten):
1. `prepare`: sample table (siCtrl libraries of the pooling sheets), gene counts (featureCounts of `nanoflowz`, all genes of `gencode.v42.basic`);
1. `prepare_cs`: cleavage sites from the read tags (`read_tags/*.tags.tsv.gz`: `XT` = gene, `XF` = cleavage site, `pt` = poly(A) tail length), one CS per position, CS annotation (site class, PAQR3 terminal exon, PolyASite atlas);
2. `sanity`: C++ Sanity (EAP) on genes and on CS, separately for Total and for the fractions (the inputs of fracSanity: each normalised on its own, the fractions add their own variability), and jointly on the CS of Total and fractions (relative usage and APA, which are ratios within a library); ISMARA replicate averages per cohort (`reml`);
3. `fit_cohorts`: fracSanity per cohort, cohort average vs Total average, for genes and for CS;
4. `fit_replicates`: fracSanity of every fraction library vs the Total average (genes), the input of the gene-set analysis;
5. `usage`: relative usage of each CS against the other CS of its gene, usage-weighted mean 3' UTR length of the CS in the terminal exon (tandem APA), most proximal vs distal CS; per cohort vs Total.

CS-level outputs are `inputs/cs_pos_*`, `sanity/cs_pos`, `fracSanity/cs_pos` and `usage_cs_pos`. An earlier version merged positions within 12 nt into one CS and also tested intronic polyadenylation; its outputs (`cs`, `usage`) are kept in the same folder but not used here.

In [ ]:
print(f"sbatch ONT_fracSanity.sbatch   # outputs in {SU_OUT}")
print(f"log: {sorted(glob.glob('ONT_fracSanity_*.log'))[-1:] }")

## Samples

siCtrl libraries of the three batches (pooling sheets, as in `ONT_organelles_vs_Total.ipynb`) and the three siCtrl Total libraries of the HCT116 run. Libraries with fewer than 100,000 reads on the nuclear protein-coding genes with ≥ 10 CPM in Total are left out (`use` = False), as in `ONT_organelles_vs_Total.ipynb`; in the ER/TIS batch, these are libraries with 88–96% mitochondrial reads.

In [ ]:
meta_all = pd.read_csv(os.path.join(SU_OUT, "inputs", "metadata.tsv"), sep="\t")
save_tsv(meta_all.drop(columns=["featurecounts", "read_tags"]), "samples", "libraries.tsv")
display(meta_all[["sample", "batch", "fraction", "cohort", "replicate", "sample_id", "assigned_reads", "reads_on_genes", "use"]])
meta = meta_all.loc[meta_all["use"]].reset_index(drop=True)

def cohort_sort_key(c):
    r = meta.loc[meta["cohort"] == c].iloc[0]
    return (BATCH_ORDER.index(r["batch"]), FRACTION_ORDER.index(r["fraction"]))
COHORTS = sorted(meta.loc[meta["fraction"] != "Total", "cohort"].unique(), key=cohort_sort_key)
cohort_info = meta.groupby("cohort")[["batch", "fraction"]].first().assign(n_libraries=meta.groupby("cohort").size())
cohort_info = cohort_info.loc[["Total"] + COHORTS]
display(cohort_info)

genes_df = pd.read_csv(os.path.join(SU_OUT, "inputs", "genes.tsv"), sep="\t", index_col=0)
gene_names = genes_df["gene_name"].fillna(pd.Series(genes_df.index, index=genes_df.index))
# biotype groups: mitochondrially encoded genes apart
biotype = genes_df["gene_type"].where(genes_df["chrom"] != "chrM", "mitochondrially_encoded")

## Sanity normalization

C++ Sanity via Sanity_utils, `v_method='EAP'`, separately on the three Total libraries and on all fraction libraries, without length correction (log2 CPM over all genes). Below: inferred biological variance $v_g$ against expression, and a PCA of the log2 CPM of genes with reads in all libraries (Total and fractions; the two Sanity runs are normalised separately, both to CPM).

In [ ]:
LEVEL_PARTS = {"genes": ("Total", "fractions"), CS_LEVEL: ("Total", "fractions", "joint")}
sanity = {(lvl, part): load_pickle(os.path.join(SU_OUT, "sanity", lvl, part, "sanity.pkl")) for lvl, parts in LEVEL_PARTS.items() for part in parts}
for (lvl, part), d in sanity.items():
    r = d["result"]
    print(f"{lvl:6s} {part:9s}: {r.log2_expression.shape[0]:7,} features x {r.log2_expression.shape[1]:2d} libraries | {r.params.get('units')} | "
          f"v_method {r.params.get('v_method')}")

sns.set(font_scale=0.9, style="ticks")
for part in ("Total", "fractions"):
    fig = su_plotting.plot_variance_vs_expression(sanity[("genes", part)]["result"])
    savefig(fig, "QC", f"variance_vs_expression.genes.{part}")
    plt.show()

# PCA: log2 CPM of genes with reads in all used libraries, Total and fractions together
gene_counts = pd.read_csv(os.path.join(SU_OUT, "inputs", "gene_counts.tsv.gz"), sep="\t", index_col=0)
expressed = gene_counts.index[(gene_counts[meta["sample"]] > 0).all(axis=1)]
X = pd.concat([sanity[("genes", p)]["result"].log2_expression for p in ("Total", "fractions")], axis=1).reindex(expressed)[meta["sample"]].dropna()
Xc = X.sub(X.mean(axis=1), axis=0).to_numpy()
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
var_expl = S ** 2 / np.sum(S ** 2)
pcs = pd.DataFrame(Vt[:3].T * S[:3], index=X.columns, columns=["PC1", "PC2", "PC3"]).join(meta.set_index("sample")[["batch", "fraction", "replicate"]])
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
for ax, (a, b) in zip(axes, (("PC1", "PC2"), ("PC1", "PC3"))):
    for s, r in pcs.iterrows():
        ax.scatter(r[a], r[b], color=FRACTION_COLORS[r["fraction"]], marker=BATCH_MARKERS[r["batch"]], s=50, edgecolor="black", lw=0.4)
        ax.annotate(r["replicate"], (r[a], r[b]), fontsize=6, xytext=(3, 3), textcoords="offset points")
    ax.set(xlabel=f"{a} ({var_expl[int(a[-1]) - 1]:.0%})", ylabel=f"{b} ({var_expl[int(b[-1]) - 1]:.0%})")
axes[1].legend(handles=notation_handles(pcs["fraction"], pcs["batch"]), fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1))
axes[0].set_title(f"{len(X):,} genes with reads in all {X.shape[1]} libraries", fontsize=9)
sns.despine(); fig.tight_layout()
savefig(fig, "QC", "PCA.genes.log2CPM")
plt.show()

# normalized expression tables: log2 CPM per library and cohort averages, with gene annotation
annot = genes_df[["gene_name", "gene_type", "chrom"]]
log2cpm = pd.concat([sanity[("genes", p)]["result"].log2_expression for p in ("Total", "fractions")], axis=1)
save_tsv(annot.join(log2cpm, how="right").rename_axis("gene_id").reset_index(), "normalized_expression", "Sanity_log2CPM.genes.per_library.tsv.gz")
avg_g = {p: sanity[("genes", p)]["averages"] for p in ("Total", "fractions")}
avg_tab = pd.concat([avg_g[p].means.add_prefix("mean;").join(avg_g[p].errors.add_prefix("error;")) for p in ("Total", "fractions")], axis=1)
save_tsv(annot.join(avg_tab, how="right").rename_axis("gene_id").reset_index(), "normalized_expression", "Sanity_log2CPM.genes.cohort_averages.tsv.gz")

### Interpretation of the Sanity normalization

- **Genes.** 25,518 genes have reads in the three Total libraries and 23,492 in the 24 fraction libraries; 21,848 have reads in both and enter fracSanity. Most of the others are weakly expressed non-coding genes.
- **The libraries order by nuclear contamination along PC1** (48% of the variance; 3,968 genes with reads in all 27 libraries). Total, the Free fractions of all three batches and TIS lie on one side, Golgi, PEX, Endo and TJ on the other, Org2 and ER in between. This is the order of the MALAT1 signal and of the compression in `ONT_organelles_vs_Total.ipynb`. The Free libraries are closest to Total.
- **The libraries of one fraction group together on PC1 across preparations,** for Golgi, PEX, Endo and TJ, but spread on PC2 (17%), e.g. the two PEX libraries; PC2 does not separate fractions.

## fracSanity: global fraction $F$ and the shape of the fraction vs Total

For each cohort, $D_g = \langle A^{\text{fraction}}_g\rangle - \langle A^{\text{Total}}_g\rangle = \log_2\alpha_g - \log_2 F + \varepsilon_g$, with $\alpha_g \sim$ Beta$(a, b)$, the contaminated-Gaussian error model with $\pi \le 0.5$, and a uniform prior on $F$; profile likelihood over $\log_2 F$, $F$ reported as posterior median with 95% interval. Genes: all genes with reads in both the Total and the fraction Sanity run.

If every gene had the same $\alpha_g$, the log2 CPM distribution of the fraction would equal that of Total up to noise; the densities below show how the shapes differ (compare the compression analysis of `ONT_organelles_vs_Total.ipynb`, which is on nuclear protein-coding genes with ≥ 10 CPM in Total).

In [ ]:
fits = {lvl: {c: load_pickle(os.path.join(SU_OUT, "fracSanity", lvl, "cohorts", safe(c) + ".pkl")) for c in COHORTS} for lvl in ("genes", CS_LEVEL)}
F_df = pd.DataFrame([{"level": lvl, "cohort": c, "batch": cohort_info.loc[c, "batch"], "fraction": cohort_info.loc[c, "fraction"],
                      "n_libraries": cohort_info.loc[c, "n_libraries"], **{k: f[k] for k in ("F", "F_CI_lower", "F_CI_upper", "F_MLE", "log2F_mean",
                      "var_log2_F", "F_identifiable", "a", "b", "pi_outlier", "tau", "ks_stat")}, "n_features": len(f["gene_ids"])}
                     for lvl in fits for c, f in fits[lvl].items()])
save_tsv(F_df, "fracSanity", "fracSanity_global_F.tsv")
display(F_df.round(4))

sns.set(font_scale=0.9, style="ticks")
fig, ax = plt.subplots(figsize=(9, 3.8))
for k, lvl in enumerate(("genes", CS_LEVEL)):
    d = F_df.loc[F_df["level"] == lvl].set_index("cohort").loc[COHORTS]
    for i, (c, r) in enumerate(d.iterrows()):
        ax.errorbar(i + (k - 0.5) * 0.3, r["F"], yerr=[[r["F"] - r["F_CI_lower"]], [r["F_CI_upper"] - r["F"]]], fmt=BATCH_MARKERS[r["batch"]],
                    color=FRACTION_COLORS[r["fraction"]], mfc=FRACTION_COLORS[r["fraction"]] if lvl == "genes" else "white", mec=FRACTION_COLORS[r["fraction"]],
                    ecolor="grey", ms=7, capsize=2)
ax.set_yscale("log")
ax.set_xticks(range(len(COHORTS)), COHORTS, rotation=60, ha="right", fontsize=8)
color_ticklabels(ax.get_xticklabels(), [cohort_info.loc[c, "fraction"] for c in COHORTS])
ax.set(ylabel="$F$ (posterior median, 95% interval)")
ax.legend(handles=[matplotlib.lines.Line2D([], [], color="grey", marker="o", ls="", label="genes"),
                   matplotlib.lines.Line2D([], [], color="grey", marker="o", mfc="white", ls="", label="cleavage sites")],
          fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1))
sns.despine(); fig.tight_layout()
savefig(fig, "fracSanity", "globalF_per_cohort")
plt.show()

# densities of log2 CPM in Total and in each cohort (genes in both)
tot_av, frac_av = sanity[("genes", "Total")]["averages"], sanity[("genes", "fractions")]["averages"]
common = tot_av.means.index.intersection(frac_av.means.index)
t_all = tot_av.means.loc[common, "Total"].to_numpy()
grid = np.linspace(-6, 18, 300)
batches = [b for b in BATCH_ORDER[1:] if (cohort_info["batch"] == b).any()]
fracs = [f for f in FRACTION_ORDER[1:] if (cohort_info["fraction"] == f).any()]
fig, axes = plt.subplots(len(batches), len(fracs), figsize=(2.3 * len(fracs), 2.2 * len(batches)), sharex=True, sharey=True, squeeze=False)
shape_rows = []
for c in COHORTS:
    b, f = cohort_info.loc[c, ["batch", "fraction"]]
    ax = axes[batches.index(b), fracs.index(f)]
    p = frac_av.means.loc[common, c].to_numpy()
    ax.plot(grid, stats.gaussian_kde(t_all)(grid), color="black", lw=1.1)
    ax.plot(grid, stats.gaussian_kde(p)(grid), color=FRACTION_COLORS[f], lw=1.6)
    ax.set_title(c, fontsize=8)
    shape_rows.append({"cohort": c, "sd_Total": t_all.std(), "sd_fraction": p.std(), "median_Total": np.median(t_all), "median_fraction": np.median(p),
                       "KS_fraction_vs_Total": stats.ks_2samp(p, t_all).statistic, "var_D": np.var(p - t_all), "spearman_D_vs_Total": spearmanr(p - t_all, t_all)[0]})
for ax in axes.flat:
    if not ax.lines:
        ax.axis("off")
for ax in axes[-1]:
    ax.set_xlabel("log2 CPM")
for ax in axes[:, 0]:
    ax.set_ylabel("density")
sns.despine(); fig.tight_layout()
savefig(fig, "fracSanity", "fraction_vs_Total.density.all_genes")
plt.show()
shape_df = pd.DataFrame(shape_rows)
save_tsv(shape_df, "fracSanity", "fraction_vs_Total.shape_stats.all_genes.tsv")
display(shape_df.round(3))

# fit diagnostics per cohort (profile likelihood, PIT histogram, alpha_g vs the fitted Beta), saved only
for lvl in ("genes", CS_LEVEL):
    for c, f in fits[lvl].items():
        fig = su_plotting.plot_frac_sanity_fit(f, condition_name=f"{c} ({lvl})")
        out = os.path.join(FIGS, "fracSanity", f"fit_diagnostics_{lvl}")
        os.makedirs(out, exist_ok=True)
        fig.savefig(os.path.join(out, f"fit.{safe(c)}.png"), bbox_inches="tight", dpi=150)
        plt.close(fig)
print("fit diagnostics:", os.path.join(FIGS, "fracSanity", "fit_diagnostics_genes"), f"and ..._{CS_LEVEL}")

### Interpretation of $F$ and of the shapes

- **$F$ is not determined by the data, as in the P-body analysis.** In 11 of 12 cohorts the profile likelihood is flat towards small $F$ (`F_identifiable` False), and the 95% intervals span about two orders of magnitude (e.g. Golgi 0.00004–0.0056). The posterior medians (0.001–0.07) follow the uniform prior and cannot be read as the share of the cell's mRNA in the sort. Spike-ins added before library preparation, or the number of sorted events with the RNA yield, would be needed to measure it.
- **ER/TIS Free ($F$ = 0.52, "identifiable") is an artefact.** It is a single library with the widest distribution of $D_g$; the fitted Beta(3.2, 1.8) puts most $\alpha_g$ near 1, so that the bound $\alpha_g \le 1$ fixes $F$, and the error scale $\tau$ is at the edge of its grid. The CS-level fit of the same library gives $F$ = 0.037 and is not identified. Its $\alpha_g$ intervals are therefore too narrow.
- **The outlier fraction is at its bound ($\pi$ = 0.5) in Org2 of the Golgi/PEX and End/TJ batches, Golgi, PEX, Endo and TJ:** the fractions where $D_g$ is most spread (var($D_g$) 1.6–2.3, against 1.3–1.7 in Free, ER and TIS). The Beta model of $\alpha_g$ describes them least well.
- **Shapes over all genes.** The fractions have a higher median log2 CPM than Total (0.8–1.7 vs 0.7) and a similar spread. The KS distance to Total is 0.07–0.09 for Free, Org2, ER and TIS, and 0.13–0.14 for Golgi, PEX, Endo and TJ. $D_g$ decreases with expression in Total in all cohorts (Spearman −0.10 to −0.32), most in Free: over all genes this mostly reflects the many weakly expressed genes of the shallow fraction libraries, whose log2 CPM Sanity shrinks towards their mean. The compression of the well-expressed genes is quantified in `ONT_organelles_vs_Total.ipynb`.

## $\alpha_g$ of individual genes: over- and under-represented mRNAs per fraction

$\alpha_g$ per cohort (posterior median, 95% interval, uncertainty of $F$ included). Because $F$ is weakly determined, genes are compared **within a cohort relative to its average gene**: $r_g = \log_2\alpha_g - \text{median}_g \log_2\alpha_g$ (posterior mean of $\log_2\alpha_g$), the same for any value of $F$. The 95% intervals of $\alpha_g$ are dominated by the uncertainty of $F$, which all genes of a cohort share (posterior s.d. of $\log_2\alpha_g$ ≈ 1.9); as in the P-body analysis, the calls therefore use the gene-specific variance, the posterior variance of $\log_2\alpha_g$ minus that of $\log_2 F$, at least the measurement variance of $D_g$, and a normal approximation: local false sign rate $\text{lfsr}_g = \Phi(-|r_g| / s_g)$. A gene is called **over-represented** in a fraction if $r_g > 0.5$ and lfsr < 0.05, **under-represented** if $r_g < -0.5$ and lfsr < 0.05; only genes with ≥ 10 CPM in Total are called. The posterior means are shrunk towards the Beta distribution of the $\alpha_g$ (slope of $r_g$ on the centred $D_g$ 0.2–0.7), so $r_g$ is smaller than the raw log2 ratio of fraction and Total.

In [ ]:
from scipy.stats import norm
MIN_CPM_CALL, R_CUT, LFSR_CUT = 10, 0.5, 0.05
total_cpm = 2 ** tot_av.means["Total"]
var_log2F = F_df.loc[F_df["level"] == "genes"].set_index("cohort")["var_log2_F"]
alpha_rows = []
for c, f in fits["genes"].items():
    a = pd.DataFrame({"gene_id": f["gene_ids"], "alpha": f["alpha_g"], "alpha_CI_lower": f["alpha_g_CI_lower"], "alpha_CI_upper": f["alpha_g_CI_upper"],
                      "log2_alpha_mean": f["log2_alpha_mean"], "log2_alpha_sd": np.sqrt(f["log2_alpha_var"])})
    a["D"] = (frac_av.means[c] - tot_av.means["Total"]).reindex(a["gene_id"]).to_numpy()
    a["r_rel"] = a["log2_alpha_mean"] - np.median(a["log2_alpha_mean"])
    a["Total_CPM"] = total_cpm.reindex(a["gene_id"]).to_numpy()
    # gene-specific s.d.: posterior variance minus the variance of log2 F (shared by all genes), at least var(D_g)
    var_D = (frac_av.errors[c] ** 2 + tot_av.errors["Total"] ** 2).reindex(a["gene_id"]).to_numpy()
    a["sd_rel"] = np.sqrt(np.maximum(a["log2_alpha_sd"] ** 2 - var_log2F[c], var_D))
    a["lfsr_rel"] = norm.cdf(-a["r_rel"].abs() / a["sd_rel"])
    sig = (a["Total_CPM"] >= MIN_CPM_CALL) & (a["lfsr_rel"] < LFSR_CUT)
    a["call"] = np.select([sig & (a["r_rel"] > R_CUT), sig & (a["r_rel"] < -R_CUT)], ["over", "under"], "n.s.")
    alpha_rows.append(a.assign(cohort=c, batch=cohort_info.loc[c, "batch"], fraction=cohort_info.loc[c, "fraction"]))
alpha_df = pd.concat(alpha_rows, ignore_index=True)
alpha_df.insert(1, "gene_name", gene_names.reindex(alpha_df["gene_id"]).to_numpy())
alpha_df.insert(2, "gene_type", biotype.reindex(alpha_df["gene_id"]).to_numpy())
save_tsv(alpha_df, "alpha_genes", "fracSanity_alpha.genes.per_cohort.tsv.gz")
n_calls = alpha_df.groupby("cohort")["call"].value_counts().unstack(fill_value=0).reindex(COHORTS)
display(n_calls)

# distribution of alpha_g per cohort with F
sns.set(font_scale=0.9, style="ticks")
fig, ax = plt.subplots(figsize=(10, 3.8))
d = alpha_df.assign(log10_alpha=np.log10(alpha_df["alpha"]))
sns.violinplot(data=d, x="cohort", y="log10_alpha", order=COHORTS, hue="fraction", palette=FRACTION_COLORS, cut=0, inner="quart", ax=ax,
               legend=False, density_norm="width")
Fg = F_df.loc[F_df["level"] == "genes"].set_index("cohort")["F"]
ax.scatter(range(len(COHORTS)), np.log10(Fg.loc[COHORTS]), marker="D", color="red", zorder=3, label="$F$")
ax.set_xticks(range(len(COHORTS)), COHORTS, rotation=60, ha="right", fontsize=8)
color_ticklabels(ax.get_xticklabels(), [cohort_info.loc[c, "fraction"] for c in COHORTS])
ax.set(xlabel="", ylabel=r"$\log_{10}\alpha_g$ (posterior median)")
ax.legend(fontsize=8, frameon=False)
sns.despine(); fig.tight_layout()
savefig(fig, "alpha_genes", "alpha_g_distribution_per_cohort")
plt.show()

# heatmap of r_g for the top over- and under-represented genes of every cohort (called genes, by r_g)
N_TOP = 6
top = []
for c in COHORTS:
    d = alpha_df.loc[alpha_df["cohort"] == c]
    top += d.loc[d["call"] == "over"].nlargest(N_TOP, "r_rel")["gene_id"].tolist() + d.loc[d["call"] == "under"].nsmallest(N_TOP, "r_rel")["gene_id"].tolist()
top = list(dict.fromkeys(top))
mat = alpha_df.pivot(index="gene_id", columns="cohort", values="r_rel").reindex(index=top, columns=COHORTS)
calls = alpha_df.pivot(index="gene_id", columns="cohort", values="call").reindex(index=top, columns=COHORTS)
save_tsv(mat.rename(index=gene_names).rename_axis("gene_name").reset_index(), "alpha_genes", "r_rel.top_genes_per_cohort.tsv")
if len(mat):
    lim = np.nanmax(np.abs(mat.values))
    col_colors = pd.DataFrame({"fraction": [FRACTION_COLORS[cohort_info.loc[c, "fraction"]] for c in COHORTS],
                               "batch": [BATCH_COLORS[cohort_info.loc[c, "batch"]] for c in COHORTS]}, index=COHORTS)
    g = sns.clustermap(mat.fillna(0).rename(index=gene_names), cmap="RdBu_r", center=0, vmin=-lim, vmax=lim, col_cluster=False, row_cluster=len(mat) > 2,
                       col_colors=col_colors, annot=np.where(calls.isin(["over", "under"]), "*", ""), fmt="", annot_kws={"fontsize": 7},
                       figsize=(0.45 * len(COHORTS) + 4, 0.16 * len(mat) + 3), yticklabels=True, dendrogram_ratio=(0.1, 0.05),
                       cbar_kws={"label": r"$r_g = \log_2\alpha_g$ - cohort median"}, cbar_pos=(-0.08, 0.6, 0.02, 0.2))
    g.ax_heatmap.set(xlabel="", ylabel="")
    g.ax_heatmap.tick_params(axis="y", labelsize=6)
    color_ticklabels(g.ax_heatmap.get_xticklabels(), [cohort_info.loc[t.get_text(), "fraction"] for t in g.ax_heatmap.get_xticklabels()])
    g.savefig(os.path.join(FIGS, "alpha_genes", "r_rel.top_genes_per_cohort.clustermap.png"), bbox_inches="tight", dpi=200)
    g.savefig(os.path.join(FIGS, "alpha_genes", "r_rel.top_genes_per_cohort.clustermap.pdf"), bbox_inches="tight")
    figure_caption(os.path.join(FIGS, "alpha_genes", "r_rel.top_genes_per_cohort.clustermap"))
    plt.show()

In [ ]:
# gene biotypes: median r_g of the biotype's genes per cohort (all genes with alpha), and the number of genes
bt = alpha_df.groupby(["gene_type", "cohort"])["r_rel"].median().unstack().reindex(columns=COHORTS)
n_bt = alpha_df.loc[alpha_df["cohort"] == COHORTS[0], "gene_type"].value_counts()
bt = bt.loc[n_bt.index[n_bt >= 20]]
bt.index = [f"{b} (n={n_bt[b]:,})" for b in bt.index]
save_tsv(bt.rename_axis("gene_type").reset_index(), "alpha_genes", "r_rel.median_per_biotype.tsv")
lim = np.nanmax(np.abs(bt.values))
sns.set(font_scale=0.8, style="white")
fig, ax = plt.subplots(figsize=(0.5 * len(COHORTS) + 3, 0.3 * len(bt) + 1.5))
sns.heatmap(bt, cmap="RdBu_r", center=0, vmin=-lim, vmax=lim, annot=True, fmt=".1f", annot_kws={"fontsize": 6}, ax=ax,
            cbar_kws={"label": "median $r_g$"})
ax.set(xlabel="", ylabel="")
color_ticklabels(ax.get_xticklabels(), [cohort_info.loc[c, "fraction"] for c in COHORTS])
savefig(fig, "alpha_genes", "r_rel.median_per_biotype")
plt.show()

# selected genes: compartment markers, nuclear RNAs, mitochondrial and housekeeping mRNAs
GOI = ["MALAT1", "NEAT1", "MT-CO1", "MT-ND1", "CALR", "HSPA5", "SSR4", "CANX", "GOLGA2", "PEX14", "EEA1", "OCLN", "TJP1", "ZFP36L1",
       "ACTB", "GAPDH", "RPLP0", "EEF1A1"]
genes_with_alpha = set(alpha_df["gene_id"])
name_to_id = {n: g for g, n in gene_names.items() if g in genes_with_alpha}
goi = [n for n in GOI if n in name_to_id]
ncols = 6
nrows = int(np.ceil(len(goi) / ncols))
sns.set(font_scale=0.8, style="ticks")
fig, axes = plt.subplots(nrows, ncols, figsize=(2.9 * ncols, 2.6 * nrows), sharey=True, squeeze=False)
for ax, n in zip(axes.flat, goi):
    d = alpha_df.loc[alpha_df["gene_id"] == name_to_id[n]].set_index("cohort").reindex(COHORTS)
    for i, (c, r) in enumerate(d.iterrows()):
        if pd.isna(r["alpha"]):
            continue
        ax.errorbar(i, r["alpha"], yerr=[[r["alpha"] - r["alpha_CI_lower"]], [r["alpha_CI_upper"] - r["alpha"]]], fmt=BATCH_MARKERS[cohort_info.loc[c, "batch"]],
                    color=FRACTION_COLORS[cohort_info.loc[c, "fraction"]], ms=5, ecolor="grey", elinewidth=0.8, capsize=1.5)
        ax.plot(i, Fg[c], marker="_", color="red", ms=8)
    ax.set_yscale("log")
    ax.set_xticks(range(len(COHORTS)), [cohort_info.loc[c, "fraction"] for c in COHORTS], rotation=90, fontsize=6)
    color_ticklabels(ax.get_xticklabels(), [cohort_info.loc[c, "fraction"] for c in COHORTS])
    ax.set_title(n, fontsize=9)
for ax in axes.flat[len(goi):]:
    ax.axis("off")
for ax in axes[:, 0]:
    ax.set_ylabel(r"$\alpha_g$ (red: $F$)")
fig.legend(handles=notation_handles([], BATCH_ORDER[1:]), loc="upper right", fontsize=7, frameon=False)
fig.tight_layout()
savefig(fig, "alpha_genes", "selected_genes.alpha_per_cohort")
plt.show()

### Interpretation of $\alpha_g$

- **The posterior $\alpha_g$ are dominated by the uncertainty of $F$;** the comparison of genes within a cohort ($r_g$) is not. The posterior means are shrunk towards the fitted Beta distribution, by a factor of 0.2–0.7 relative to the centred $D_g$.
- **Few mRNAs are called in Free, Org2, ER and TIS** (ER/TIS: 1–19 over- and 78–226 under-represented genes; Golgi/PEX Free 2 / 69; End/TJ Free 60 / 525), **many in Golgi and Endo** (374 / 1,706 and 363 / 1,635). PEX has 0 / 184 and TJ 4 / 658 (the shrinkage is strongest where $\pi$ is at its bound). Under-represented genes outnumber over-represented ones everywhere, because the distribution of $\alpha_g$ is skewed and the posterior pulls high values in more than low ones.
- **Under-represented in all fractions: small non-coding RNAs and their host transcripts** (snoRNAs such as SNORD22/26/27, miRNA genes, RMRP, the snoRNA and miRNA biotypes in the biotype heatmap): RNAs that are processed or retained in the nucleus and whose poly(A) reads come from precursors, depleted from every sorted fraction relative to whole cells. The median $r_g$ of protein-coding genes is slightly positive (+0.01 to +0.14) and that of lncRNAs negative (−0.01 to −0.24) in every cohort, both most in Golgi, PEX, Endo and TJ.
- **Mitochondrially encoded genes** are under-represented in the ER/TIS Free and TIS libraries and in Free of the other batches (median $r_g$ −0.4 to −1.2) and over-represented in PEX and TJ (+0.7 to +0.85): this is the mitochondrial content of the sorts (QC in `ONT_organelles_vs_Total.ipynb`). Over-represented "genes" in Org2 and TJ include nuclear mitochondrial pseudogenes (MTND1P23, MTND2P28), most likely mitochondrial reads mapped to their nuclear copies.
- **Over-represented in Golgi and Endo:** e.g. TRIM25, GFPT1, RBCK1, PYGB, SLC7A5, FTH1, MYH9, FLNB; the functional groups behind them are in the next section.
- **Selected genes ($r_g$).** MALAT1 and NEAT1 are highest in Golgi, PEX and TJ (+0.7 to +1.3) and lowest in Free (−0.15 to −0.8). The mRNAs of ER-resident proteins (CALR, HSPA5, CANX) are depleted from the Free fraction of every batch (−0.3 to −1.0), as expected for mRNAs translated at the ER, and enriched in Golgi, Endo, TJ and PEX (+0.4 to +1.2), but not in the ER (CANX) sort itself (−0.02 to +0.13). EEA1 mRNA is highest in Endo (+0.70), but OCLN mRNA is also highest in Endo (+0.84, TJ +0.55), and PEX14 mRNA is not enriched in PEX (−0.44): the sorts enrich organelles by a protein marker, which need not enrich the marker's own mRNA. GAPDH is under-represented in most fractions (to −2.1).

## Functional groups over- and under-represented in each fraction (ISMARA)

ISMARA-style activities fitted on the **per-library** $\log_2\alpha_g$ (posterior means of fracSanity fits of each fraction library against the Total average; replicates are not paired with Total libraries). Columns are centred (`center="columns"`): the activity of a set in a library is the shift of the $\log_2\alpha$ of its genes beyond the average gene of that library, independent of $F$. Activities are averaged over the libraries of a cohort (`reml`), and each cohort's average is tested against 0 (`activity_contrast`): **positive = the set's mRNAs are over-represented in the fraction** relative to the average mRNA, negative = under-represented.

As in `pbody_RNA_Spang.fracSanity.pcTPM.ipynb`, **one fit per database** (literature, GO:BP, GO:CC, GO:MF, Reactome, KEGG, WikiPathways; 5–500 genes per set, identical sets merged), each with two kinds of covariates: the gene biotypes (vs protein-coding genes) and the mitochondrially encoded genes (`BIOTYPE:`), and the deciles of expression in Total (`EXPR:decile01` = lowest, decile 5 as reference). The fractions are compressed against Total (`ONT_organelles_vs_Total.ipynb`): their $\log_2\alpha_g$ depends on how abundant the mRNA is in the cell, and without the expression covariates, sets of abundant mRNAs (ribosomal proteins, translation, respiration) would appear under-represented for that reason alone. With them, a set's activity is its over- or under-representation **beyond what its genes' abundance and biotype predict**. The covariate rows are reported from the literature fit. `padj` is BH over the sets of one fit and cohort. Gene IDs: Ensembl gene IDs without version (GO symbols, KEGG and WikiPathways NCBI IDs translated with the NCBI `gene_info` and the annotation). Literature sets: cytoplasmic ribosomal-protein genes (`LIT:RP`) and replication-dependent histone genes (`LIT:histone_RD`, non-polyadenylated mRNAs).

In [ ]:
# per-library log2 alpha (posterior means), genes x libraries
lib_fits = {}
for s in meta.loc[meta["fraction"] != "Total", "sample"]:
    lib_fits[s] = load_pickle(os.path.join(SU_OUT, "fracSanity", "genes", "libraries", safe(s) + ".pkl"))
la = pd.DataFrame({s: pd.Series(f["log2_alpha_mean"], index=f["gene_ids"]) for s, f in lib_fits.items()})
lib_meta = meta.loc[meta["sample"].isin(la.columns), ["sample", "cohort"]].reset_index(drop=True)
F_lib = pd.DataFrame([{"sample": s, "cohort": f["cohort"], "F": f["F"], "F_CI_lower": f["F_CI_lower"], "F_CI_upper": f["F_CI_upper"],
                       "pi_outlier": f["pi_outlier"], "a": f["a"], "b": f["b"]} for s, f in lib_fits.items()])
save_tsv(F_lib, "gene_sets", "fracSanity_global_F.per_library.tsv")
la_out = la.copy()
la_out.insert(0, "gene_name", gene_names.reindex(la_out.index).to_numpy())
save_tsv(la_out.rename_axis("gene_id").reset_index(), "gene_sets", "fracSanity_log2_alpha.genes.per_library.tsv.gz")

# Ensembl IDs without version (PAR_Y copies removed)
values = la.loc[~la.index.str.endswith("_PAR_Y")]
values.index = values.index.str.replace(r"\.\d+$", "", regex=True)
assert values.index.is_unique
universe = set(values.index)
ens_of = pd.Series(genes_df.index[~genes_df.index.str.endswith("_PAR_Y")].str.replace(r"\.\d+$", "", regex=True),
                   index=genes_df.index[~genes_df.index.str.endswith("_PAR_Y")])
nm = gene_names.reindex(ens_of.index)
sym2ens = pd.Series(ens_of.to_numpy(), index=nm.to_numpy()).groupby(level=0).agg(list).to_dict()
gene_info = read_ncbi_gene_info(next(latest_release(GENESETS_DIR, "kegg").glob("*.gene_info*")))
ncbi2ens = {}
for gid, sym, xref in zip(gene_info["GeneID"].astype(str), gene_info["Symbol"], gene_info["dbXrefs"].fillna("")):
    e = [x.split(":", 1)[1] for x in xref.split("|") if x.startswith("Ensembl:")]
    ncbi2ens[gid] = e if e else sym2ens.get(sym, [])
ncbi2ens = {k: v for k, v in ncbi2ens.items() if v}

def prepare(coll):
    return coll.restrict_to(universe).filter_size(5, 500).drop_duplicates()

ens_name = pd.Series(nm.to_numpy(), index=ens_of.to_numpy())
lit = {"LIT:RP": {g for g, n in ens_name.items() if re.match(r"^RP[LS]\d+[A-Z]?\d*$", str(n)) and g in universe},
       "LIT:histone_RD": {g for g, n in ens_name.items() if re.match(r"^H(1-[0-6]|2AC\d+|2BC\d+|3C\d+|4C\d+)$", str(n)) and g in universe}}
db_colls = {"literature": GeneSetCollection(lit, {"LIT:RP": "cytoplasmic ribosomal-protein genes",
                                                  "LIT:histone_RD": "replication-dependent histone genes"}, "literature")}
for aspect in ("BP", "CC", "MF"):
    db_colls[f"GO:{aspect}"] = prepare(load_go(GENESETS_DIR, aspect=aspect, id_type="symbol").map_ids(sym2ens))
db_colls["Reactome"] = prepare(load_reactome(GENESETS_DIR))
db_colls["KEGG"] = prepare(load_kegg(GENESETS_DIR, id_map=ncbi2ens))
db_colls["WikiPathways"] = prepare(load_wikipathways(GENESETS_DIR, id_map=ncbi2ens))

bt_of = biotype.loc[ens_of.index]
bt_of.index = ens_of.to_numpy()
biotype_sets = {f"BIOTYPE:{b}": set(g.index) & universe for b, g in bt_of.groupby(bt_of) if b != "protein_coding"}
biotype_sets = {k: v for k, v in biotype_sets.items() if len(v) >= 5}
biotype_coll = GeneSetCollection(biotype_sets, {k: k.split(":")[1].replace("_", " ") + " (biotype)" for k in biotype_sets}, "biotype")
# expression covariates: deciles of the log2 CPM in Total (decile 5 is the reference and has no set)
tot_l2 = tot_av.means["Total"].loc[lambda x: ~x.index.str.endswith("_PAR_Y")]
tot_l2.index = tot_l2.index.str.replace(r"\.\d+$", "", regex=True)
decile = pd.qcut(tot_l2.reindex(sorted(universe)).dropna(), 10, labels=False)
expr_sets = {f"EXPR:decile{d + 1:02d}": set(decile.index[decile == d]) for d in range(10) if d != 4}
expr_coll = GeneSetCollection(expr_sets, {k: f"expression in Total, decile {k[-2:]}" for k in expr_sets}, "expression")
covariate_coll = biotype_coll + expr_coll
print(f"{values.shape[1]} libraries, {len(universe):,} genes; sets per database:", {k: len(v) for k, v in db_colls.items()},
      "| covariates:", len(covariate_coll))

In [ ]:
GENESET_DATABASES = list(db_colls)
act_rows, info_all, zm_all, lib_acts = [], [], [], []
for db, c_db in db_colls.items():
    coll_db = c_db + covariate_coll
    a = su.fit_set_activities(values, coll_db, center="columns", ridge="cv", cv_repeats=5)
    a_avg = a.average_conditions(lib_meta, sample_col="sample", cond_col="cohort", method="reml")
    info = set_source_table(coll_db).assign(fit=db)
    info["size"] = info["set_id"].map(coll_db.sizes)
    info["name"] = info["set_id"].map(coll_db.names)
    info_all.append(info)
    zm_all.append(info.assign(z_m=info["set_id"].map(a.z_values)))
    lib_acts.append(a.activities.assign(fit=db))
    for c in COHORTS:
        d = su.activity_contrast(a_avg, {c: 1.0})
        act_rows.append(d[["delta_activity", "SE", "z_score", "p_value", "padj", "lfsr"]].assign(cohort=c, fit=db).rename_axis("set_id").reset_index())
    print(f"{db}: {len(c_db)} sets (+ {len(covariate_coll)} covariates); lambda = {a.lambda_:.4g}; "
          f"median explained variance per library {a.explained_variance.median():.3f}")

COVARIATE = ("BIOTYPE:", "EXPR:")
keep = lambda df: df.loc[~df["set_id"].str.startswith(COVARIATE) | (df["fit"] == "literature")].reset_index(drop=True)
info_all = keep(pd.concat(info_all))
act_df = keep(pd.concat(act_rows)).rename(columns={"delta_activity": "activity"}).merge(info_all[["set_id", "fit", "database", "name", "size"]],
                                                                                         on=["set_id", "fit"], how="left")
act_df["database"] = np.select([act_df["set_id"].str.startswith("BIOTYPE:"), act_df["set_id"].str.startswith("EXPR:")], ["biotype", "expression"],
                               act_df["fit"])
zm_df = keep(pd.concat(zm_all)).sort_values("z_m", ascending=False).reset_index(drop=True)
save_tsv(act_df, "gene_sets", "gene_set_activities.per_cohort.tsv.gz")
save_tsv(zm_df, "gene_sets", "gene_set_activities.z_m.tsv")
lib_act_df = keep(pd.concat(lib_acts).rename_axis("set_id").reset_index())
save_tsv(lib_act_df, "gene_sets", "gene_set_activities.per_library.tsv.gz")
print(act_df.loc[act_df["padj"] < 0.05].groupby(["database", "cohort"]).size().unstack(fill_value=0).reindex(columns=COHORTS).to_string())

In [ ]:
set_short = act_df.drop_duplicates("set_id").set_index("set_id")["name"].fillna("").str.slice(0, 65)
col_colors = pd.DataFrame({"fraction": [FRACTION_COLORS[cohort_info.loc[c, "fraction"]] for c in COHORTS],
                           "batch": [BATCH_COLORS[cohort_info.loc[c, "batch"]] for c in COHORTS]}, index=COHORTS)

def activity_clustermap(mat, padj, title, name, row_cluster=True, row_labels=None):
    # activities with * for padj < 0.05; columns: cohorts in batch order, colour bars for fraction and batch
    mat = mat.fillna(0)
    stars = np.where(padj.reindex(index=mat.index, columns=mat.columns) < 0.05, "*", "")
    h = 0.2 * len(mat) + 3
    lim = np.nanmax(np.abs(mat.values))
    sns.set(font_scale=0.8, style="white")
    g = sns.clustermap(mat.rename(index=row_labels or {}), cmap="RdBu_r", center=0, vmin=-lim, vmax=lim, row_cluster=row_cluster and len(mat) > 2,
                       col_cluster=False, method="average", col_colors=col_colors.reindex(mat.columns), colors_ratio=(0.03, 0.3 / h),
                       annot=stars, fmt="", annot_kws={"fontsize": 8}, figsize=(0.45 * mat.shape[1] + 7, h), yticklabels=True,
                       cbar_kws={"label": "activity (over- / under-representation)"}, dendrogram_ratio=(0.08, 0.6 / h),
                       cbar_pos=(-0.08, 0.6, 0.02, min(0.25, 2 / h)))
    g.ax_heatmap.set(xlabel="", ylabel="")
    color_ticklabels(g.ax_heatmap.get_xticklabels(), [cohort_info.loc[t.get_text(), "fraction"] for t in g.ax_heatmap.get_xticklabels()])
    g.ax_col_dendrogram.set_title(title, fontsize=9)
    out = os.path.join(FIGS, "gene_sets")
    os.makedirs(out, exist_ok=True)
    g.savefig(os.path.join(out, f"{name}.pdf"), bbox_inches="tight")
    g.savefig(os.path.join(out, f"{name}.png"), bbox_inches="tight", dpi=200)
    figure_caption(os.path.join(out, name))
    plt.show()

# per database: sets with padj < 0.05 in at least one cohort (at most MAX_ROWS, smallest padj), rows clustered
MAX_ROWS = 50
for db in GENESET_DATABASES + ["biotype", "expression"]:
    d = act_df.loc[act_df["database"] == db]
    sig = d.loc[d["padj"] < 0.05, "set_id"].unique()
    if len(sig) == 0:
        print(f"{db}: no set with padj < 0.05")
        continue
    mat = d.loc[d["set_id"].isin(sig)].pivot(index="set_id", columns="cohort", values="activity")[COHORTS]
    padj = d.loc[d["set_id"].isin(sig)].pivot(index="set_id", columns="cohort", values="padj")[COHORTS]
    save_tsv(mat.rename_axis("set_id").reset_index().assign(name=lambda x: x["set_id"].map(set_short)),
             "gene_sets", f"gene_set_activities.sig_in_any_cohort.{db.replace(':', '_')}.tsv")
    shown = padj.min(axis=1).sort_values().index[:MAX_ROWS]
    activity_clustermap(mat.loc[shown], padj.loc[shown], f"{db}: {len(shown)} of {len(sig)} sets with padj < 0.05 in at least one cohort",
                        f"activities.sig_in_any_cohort.{db.replace(':', '_')}", row_labels=set_short.to_dict())

### Sets ordered by their overall significance $z_m$

Per database, the 12 sets with the highest $z_m$ over all libraries (rows sorted by $z_m$); colour: cohort-averaged activity, \* padj < 0.05.

In [ ]:
N_TOP_ZM = 12
for db in [d for d in GENESET_DATABASES if d != "literature"]:
    top_zm = zm_df.loc[zm_df["fit"] == db].loc[lambda x: ~x["set_id"].str.startswith(COVARIATE)].head(N_TOP_ZM)
    if top_zm.empty:
        continue
    d = act_df.loc[act_df["set_id"].isin(top_zm["set_id"]) & (act_df["fit"] == db)]
    mat = d.pivot(index="set_id", columns="cohort", values="activity").reindex(index=top_zm["set_id"], columns=COHORTS)
    padj = d.pivot(index="set_id", columns="cohort", values="padj").reindex(index=top_zm["set_id"], columns=COHORTS)
    labels = {s: f"{set_short.get(s, s)}  z_m={z:.1f}" for s, z in zip(top_zm["set_id"], top_zm["z_m"])}
    activity_clustermap(mat, padj, f"{db}: top {len(mat)} sets by $z_m$", f"activities.top_z_m.{db.replace(':', '_')}", row_cluster=False, row_labels=labels)

### Interpretation of the gene-set analysis

- **Abundance explains much of the signal.** The expression covariates have the largest activities: the most abundant decile of mRNAs is under-represented in every fraction (−0.2 to −0.5 in Free, Org2, ER and TIS; −0.6 to −1.1 in Golgi, PEX, Endo and TJ) and the least abundant decile over-represented (up to +0.23). This is the compression of the fractions (`ONT_organelles_vs_Total.ipynb`) in the language of the set model. The activities below are beyond abundance and biotype.
- **Far more sets are over- or under-represented in Golgi, PEX, Endo and TJ** (e.g. 15–29 GO:CC and 26–61 KEGG sets at padj < 0.05) than in Free, Org2, ER and TIS of the ER/TIS batch (0–11 per database); Free and Org2 of the Golgi/PEX and End/TJ batches are in between (e.g. 12–39 KEGG sets).
- **mRNAs of the secretory pathway separate the Free fraction from the membrane sorts.** The mRNAs of ER protein complexes (`GO:0140534`) and ER-lumen proteins (`GO:0005788`) are under-represented in the Free fraction of all three batches and in TIS (−0.03 to −0.12), and over-represented in Golgi, PEX, Endo, TJ and Org2 of the Golgi/PEX batch (+0.08 to +0.19), as expected for mRNAs translated at the ER membrane that co-sort with membranes. They are not enriched in the ER (CANX) sort itself (+0.02 to +0.03, n.s.; two shallow libraries).
- **Ribosomal-protein mRNAs go the other way:** over-represented in Free and TIS (+0.11 to +0.18, beyond their abundance) and under-represented in PEX, Endo and TJ (−0.16 to −0.22).
- **Shared by Golgi, PEX, Endo and TJ (and partly by Org2 and Free of their batches):** endocytosis (KEGG, +0.17 to +0.22; also +0.05 to +0.12 in Free and Org2), early endosome, transcription regulator complexes, ubiquitin ligase complexes, MAPK signalling, RHO GTPase cycle and cadherin signalling are over-represented; replication-dependent histone mRNAs (non-polyadenylated; their poly(A) reads come from a minor polyadenylated form) are under-represented, and glycolysis in Endo, TJ and Free and Org2 of the End/TJ batch.
- **Organelle identity is not visible in the mRNA sets of the sorts:** peroxisome (`GO:0005777`) is not enriched in PEX and the tight-junction set not in TJ; Golgi, PEX, Endo and TJ share their set profiles, as they share nuclear contamination and the APA shift. Differences between these four sorts are smaller than their common difference from Free.
- **Biotypes:** lncRNAs (−0.3 to −0.4) and snoRNA host genes (−0.73 to −0.92) are under-represented relative to protein-coding genes most in Golgi, PEX, Endo and TJ, although MALAT1 and NEAT1 themselves are enriched there; mitochondrially encoded transcripts follow the mitochondrial content of the sorts (+1.3 to +1.5 in PEX and TJ, −0.3 to −0.6 in Free and TIS).

## Cleavage sites

From the read tags of `nanoflowz` (reads after UMI deduplication with an estimated poly(A) tail, `pt` > 0, assigned to one gene, `XT`), the cleavage site `XF` of every read. As in `CD47_analysis.ipynb`, a cleavage site is identified by gene and position (`<XT>_<XF>`), and **every position is its own CS**: nearby positions are not merged, because small differences of the 3' end can be biological (e.g. tied to poly(A) tail length or splice isoforms), and because a locus usually has one dominant position whose usage should not be mixed with that of its minor neighbours. Minor positions have few reads and get log2 expression close to their prior after Sanity, which is the honest estimate. CS with at least 5 reads over the analysed libraries enter the analyses.

**Site classes** (GENCODE v42 basic; overview only): `terminal` = in the last exon of the Ensembl canonical transcript (or up to 5 kb downstream of its annotated end) and downstream of its stop codon, `exonic` = in another exon of the gene (any transcript), `intronic` = in the gene but in no exon, `outside` = outside the annotated gene. `in_PolyASite_atlas`: within 10 nt of a PolyASite 3.0 cluster.

**Terminal exon (tandem APA).** The APA analysis uses only the CS in the **terminal exon of the gene from the PAQR3 segmentation** (`APA_AI_data/paqr3_segments_results_atlasv3_min77`: the segment with the largest index of the gene; 14,140 genes, protein-coding; the CS must be on the strand of the segment and within it or up to 10 nt past its 3' end). The 3' UTR length of such a CS is its distance from the stop codon of the Ensembl canonical transcript (+1) when the stop codon lies in the terminal exon (most genes), otherwise from the 5' end of the terminal exon; only the differences between the CS of a gene matter for the index.

Below, to show how read 3' ends scatter: positions of one gene within 12 nt of each other are grouped (single linkage, for this figure only), and the share of the group's reads at its most used position is shown.

In [ ]:
cs_tab = pd.read_csv(os.path.join(SU_OUT, "inputs", f"{CS_LEVEL}_table.tsv.gz"), sep="\t", index_col=0)
cs_counts = pd.read_csv(os.path.join(SU_OUT, "inputs", f"{CS_LEVEL}_counts.tsv.gz"), sep="\t", index_col=0)   # CS with >= 5 reads
used = cs_tab.loc[cs_tab["passes_min_reads"]]
CLASS_ORDER = ["terminal", "exonic", "intronic", "outside"]
CLASS_COLORS = {"terminal": "tab:blue", "exonic": "tab:green", "intronic": "tab:red", "outside": "lightgrey"}
summ = used.groupby("site_class").agg(n_CS=("reads", "size"), reads=("reads", "sum"), in_atlas=("in_PolyASite_atlas", "mean"),
                                      genes=("gene_id", "nunique"), in_PAQR3_terminal_exon=("in_terminal_exon", "sum")).reindex(CLASS_ORDER)
summ["share_of_reads"] = summ["reads"] / summ["reads"].sum()
print(f"{len(cs_tab):,} CS in {cs_tab['gene_id'].nunique():,} genes; {len(used):,} CS with >= 5 reads in {used['gene_id'].nunique():,} genes "
      f"({used['reads'].sum() / cs_tab['reads'].sum():.1%} of the reads)")
display(summ.round(3))
save_tsv(used.reset_index(), "cleavage_sites", "cs_table.min5reads.tsv.gz")

# share of reads per site class in every library
lib_order = [s for c in ["Total"] + COHORTS for s in meta.loc[meta["cohort"] == c, "sample"]]
share = cs_counts.loc[used.index, lib_order].groupby(used["site_class"]).sum()
share = (share / share.sum()).reindex(CLASS_ORDER).T
te_ok = used.loc[used["in_terminal_exon"]]          # CS of the tandem APA analysis
n_term = te_ok.groupby("gene_id").size()
print(f"PAQR3 terminal exons: {len(te_ok):,} CS in {te_ok['gene_id'].nunique():,} genes, {(n_term >= 2).sum():,} genes with >= 2 CS; "
      f"3' UTR length from the stop codon for {(te_ok['te_utr_origin'] == 'stop_codon').mean():.1%} of the CS")

# scatter of read 3' ends: positions of a gene within 12 nt grouped (single linkage; for this overview only)
p = cs_tab.sort_values(["gene_id", "strand", "cs_pos"])
grp_id = ((p["gene_id"] != p["gene_id"].shift()) | (p["strand"] != p["strand"].shift()) | (p["cs_pos"].diff() >= 12)).cumsum()
g = p.groupby(grp_id.to_numpy())["reads"]
loci = pd.DataFrame({"reads": g.sum(), "n_positions": g.size(), "top_share": g.max() / g.sum()})
loci = loci.loc[loci["reads"] >= 20]
print(f"{len(loci):,} groups of positions with >= 20 reads: median {loci['n_positions'].median():.0f} positions, median share of the reads "
      f"at the most used position {loci['top_share'].median():.2f} (>= 0.5 in {(loci['top_share'] >= 0.5).mean():.0%} of the groups)")
sns.set(font_scale=0.85, style="ticks")
fig, axes = plt.subplots(1, 4, figsize=(19, max(4, 0.16 * len(lib_order))), gridspec_kw={"width_ratios": [2.4, 1, 1, 1]})
left = np.zeros(len(share))
for k in CLASS_ORDER:
    axes[0].barh(np.arange(len(share)), share[k], left=left, color=CLASS_COLORS[k], height=0.85, label=k)
    left += share[k].to_numpy()
axes[0].set_yticks(np.arange(len(share)), share.index, fontsize=6.5)
color_ticklabels(axes[0].get_yticklabels(), meta.set_index("sample").loc[share.index, "fraction"])
axes[0].set(xlim=(0, 1), ylim=(len(share) - 0.5, -0.5), xlabel="share of reads at CS")
axes[0].legend(fontsize=7, frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=4)
axes[1].hist(np.log10(te_ok["utr_length"]), bins=50, color="tab:blue")
axes[1].set(xlabel="log10 3' UTR length, CS in the\nPAQR3 terminal exon (nt)", ylabel="CS")
axes[2].bar(*np.unique(np.clip(n_term, 1, 15), return_counts=True), color="tab:blue")
axes[2].set(xlabel="CS per terminal exon (15 = 15 or more)", ylabel="genes")
axes[3].hist(loci["top_share"], bins=40, color="grey")
axes[3].set(xlabel="share of reads at the most used position\n(positions within 12 nt, groups >= 20 reads)", ylabel="groups")
sns.despine(); fig.tight_layout()
savefig(fig, "cleavage_sites", "cs_classes_per_library.utr_length.n_sites")
plt.show()

## Relative usage of cleavage sites

Joint Sanity run (EAP) on the CS counts of the Total and fraction libraries. For every CS (position) of a gene with at least two CS: log2 odds of its usage against the other CS of the gene, per library (delta method), averaged per cohort (`reml`); usage $= 2^Y / (1 + 2^Y)$. Each cohort is compared with Total (`differential_relative_usage`, with the uncertainty of the CS means by Monte Carlo): $\Delta\Delta$ = change of the log2 odds, BH-adjusted p-values. Tables: usage of every CS per cohort, and the tests per cohort.

In [ ]:
usage = load_pickle(os.path.join(SU_OUT, f"usage_{CS_LEVEL}", "cs_usage.per_sample.pkl"))
u_means, u_errors, u_tau = usage["averages"]          # average_replicates: means, errors, tau (CS x cohorts)
u_tab = used[["gene_id", "gene_name", "site_class", "in_terminal_exon", "utr_length", "cs_pos", "strand", "in_PolyASite_atlas"]].join(
    (2 ** u_means / (1 + 2 ** u_means)).add_prefix("usage;"), how="inner").join(u_means.add_prefix("log2_odds;")).join(
    u_errors.add_prefix("SE;"))
save_tsv(u_tab.rename_axis("cs_id").reset_index(), "cs_usage", "cs_relative_usage.per_cohort.tsv.gz")
print(f"{len(u_tab):,} CS in {u_tab['gene_id'].nunique():,} genes with >= 2 CS")

du = {}
for c in COHORTS:
    d = pd.read_csv(os.path.join(SU_OUT, f"usage_{CS_LEVEL}", f"cs_usage.{safe(c)}_vs_Total.tsv.gz"), sep="\t", index_col=0)
    du[c] = d.join(used[["site_class", "utr_length"]])
du_long = pd.concat([d.rename(columns=lambda x: x.replace(f"_{c}", "_fraction")).assign(cohort=c) for c, d in du.items()]).rename_axis("cs_id").reset_index()
du_long.insert(2, "gene_name", gene_names.reindex(du_long["gene"]).to_numpy())
save_tsv(du_long, "cs_usage", "cs_relative_usage.fraction_vs_Total.tsv.gz")

# number of CS used more / less in the fraction than in Total (padj < 0.05), by site class
rows = []
for c, d in du.items():
    for k in CLASS_ORDER:
        s = d.loc[d["site_class"] == k]
        rows.append({"cohort": c, "site_class": k, "n_tests": len(s), "higher": int(((s["padj"] < 0.05) & (s["dd_log2FC"] > 0)).sum()),
                     "lower": int(((s["padj"] < 0.05) & (s["dd_log2FC"] < 0)).sum())})
n_du = pd.DataFrame(rows)
save_tsv(n_du, "cs_usage", "cs_relative_usage.fraction_vs_Total.n_significant.tsv")
display(n_du.pivot(index="cohort", columns="site_class", values=["higher", "lower"]).reindex(COHORTS))

sns.set(font_scale=0.85, style="ticks")
fig, axes = plt.subplots(1, len(CLASS_ORDER), figsize=(3.4 * len(CLASS_ORDER), 3.8), sharey=False)
for ax, k in zip(axes, CLASS_ORDER):
    d = n_du.loc[n_du["site_class"] == k].set_index("cohort").reindex(COHORTS)
    x = np.arange(len(COHORTS))
    cols = [FRACTION_COLORS[cohort_info.loc[c, "fraction"]] for c in COHORTS]
    ax.bar(x, d["higher"], color=cols, edgecolor="black", lw=0.4)
    ax.bar(x, -d["lower"], color=cols, edgecolor="black", lw=0.4, hatch="//")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xticks(x, COHORTS, rotation=90, fontsize=7)
    color_ticklabels(ax.get_xticklabels(), [cohort_info.loc[c, "fraction"] for c in COHORTS])
    ax.set_title(f"{k} CS", fontsize=9)
axes[0].set_ylabel("# CS used more (up) / less (down)\nthan in Total (padj < 0.05)")
sns.despine(); fig.tight_layout()
savefig(fig, "cs_usage", "cs_relative_usage.fraction_vs_Total.n_significant")
plt.show()

## 3' UTR length (tandem APA) in the fractions relative to Total

Only the CS in the PAQR3 terminal exon of a gene (see above), for genes with at least two such CS: the usage-weighted mean 3' UTR length $L_g$ of each library (`utr_length_index`), averaged per cohort; $D_g = \langle\log_2 L_g\rangle_{\text{fraction}} - \langle\log_2 L_g\rangle_{\text{Total}}$ with errors from the CS posteriors, the replicate averages and the CS means (`differential_utr_length`), call shortening / lengthening at padj < 0.05. Complementary two-site view: the distal CS against the most proximal one (`differential_proximal_distal`, `splits="first"`; $\Delta\Delta > 0$ = more distal usage in the fraction). Intronic polyadenylation is not analysed here (the assignment of intronic 3' ends to transcript isoforms by `nanoflowz` is to be checked first).

In [ ]:
apa, pdist = {}, {}
for c in COHORTS:
    apa[c] = pd.read_csv(os.path.join(SU_OUT, f"usage_{CS_LEVEL}", f"utr_length.{safe(c)}_vs_Total.tsv.gz"), sep="\t", index_col=0)
    pdist[c] = pd.read_csv(os.path.join(SU_OUT, f"usage_{CS_LEVEL}", f"proximal_vs_distal.{safe(c)}_vs_Total.tsv.gz"), sep="\t", index_col=0)
ren = lambda d, c: d.rename(columns=lambda x: x.replace(f"_{c}", "_fraction"))
apa_long = pd.concat([ren(d, c).assign(cohort=c) for c, d in apa.items()]).rename_axis("gene_id").reset_index()
apa_long.insert(1, "gene_name", gene_names.reindex(apa_long["gene_id"]).to_numpy())
save_tsv(apa_long, "apa", "utr_length.fraction_vs_Total.tsv.gz")
save_tsv(pd.concat([ren(d, c).assign(cohort=c) for c, d in pdist.items()]).rename_axis("group").reset_index(), "apa",
         "proximal_vs_distal.fraction_vs_Total.tsv.gz")
uli = load_pickle(os.path.join(SU_OUT, f"usage_{CS_LEVEL}", "utr_length_index.per_sample.pkl"))
li_means, li_errors, _ = uli["averages"]
li = li_means.add_prefix("log2_L;").join(li_errors.add_prefix("SE;"))
li.insert(0, "gene_name", gene_names.reindex(li.index).to_numpy())
save_tsv(li.rename_axis("gene_id").reset_index(), "apa", "utr_length_index.log2_mean_3UTR_length.per_cohort.tsv.gz")

summary = []
for c in COHORTS:
    d, p = apa[c], pdist[c]
    pp = p.join(d["log2FC_length"], on="gene")
    summary.append({"cohort": c, "genes": len(d), "shortening": int((d["call"] == "shortening").sum()), "lengthening": int((d["call"] == "lengthening").sum()),
                    "mean_log2FC_length": d["log2FC_length"].mean(), "median_log2FC_length": d["log2FC_length"].median(),
                    "median_log2FC_length_padj<0.05": d.loc[d["padj"] < 0.05, "log2FC_length"].median(),
                    "spearman_length_vs_distal_proximal": spearmanr(pp["log2FC_length"], pp["dd_log2FC"], nan_policy="omit")[0]})
apa_summary = pd.DataFrame(summary).set_index("cohort")
save_tsv(apa_summary.reset_index(), "apa", "apa_summary.fraction_vs_Total.tsv")
display(apa_summary.round(3))

sns.set(font_scale=0.85, style="ticks")
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1, 1.4]})
x = np.arange(len(COHORTS))
cols = [FRACTION_COLORS[cohort_info.loc[c, "fraction"]] for c in COHORTS]
ax = axes[0]
ax.bar(x, -apa_summary.loc[COHORTS, "shortening"], color=cols, edgecolor="black", lw=0.4, hatch="//", label="shortening")
ax.bar(x, apa_summary.loc[COHORTS, "lengthening"], color=cols, edgecolor="black", lw=0.4, label="lengthening")
ax.axhline(0, color="black", lw=0.8)
ax.set(ylabel="# genes: lengthening (up) /\nshortening (down) vs Total, padj < 0.05")
ax = axes[1]
d = apa_long.loc[apa_long["cohort"].isin(COHORTS)]
sns.boxplot(data=d, x="cohort", y="log2FC_length", order=COHORTS, hue="cohort", palette=dict(zip(COHORTS, cols)), showfliers=False, ax=ax, legend=False,
            width=0.6)
ax.axhline(0, color="black", lw=0.8)
ax.set(ylabel="$D_g$ = log2 mean 3' UTR length,\nfraction - Total (all tested genes)", xlabel="")
for ax in axes:
    ax.set_xticks(x, COHORTS, rotation=90, fontsize=7)
    color_ticklabels(ax.get_xticklabels(), [cohort_info.loc[c, "fraction"] for c in COHORTS])
sns.despine(); fig.tight_layout()
savefig(fig, "apa", "apa.fraction_vs_Total.summary")
plt.show()

# genes called in many cohorts: D_g heatmap
calls = apa_long.pivot(index="gene_id", columns="cohort", values="call").reindex(columns=COHORTS)
n_called = (calls != "NS").sum(axis=1)
top_genes = n_called.sort_values(ascending=False).index[:40]
mat = apa_long.pivot(index="gene_id", columns="cohort", values="log2FC_length").reindex(index=top_genes, columns=COHORTS)
if len(mat) and n_called.max() > 0:
    lim = np.nanmax(np.abs(mat.values))
    sns.set(font_scale=0.8, style="white")
    fig, ax = plt.subplots(figsize=(0.45 * len(COHORTS) + 3, 0.18 * len(mat) + 1.5))
    sns.heatmap(mat.rename(index=gene_names), cmap="RdBu_r", center=0, vmin=-lim, vmax=lim, ax=ax, yticklabels=True,
                annot=np.where(calls.reindex(index=top_genes) != "NS", "*", ""), fmt="", annot_kws={"fontsize": 7},
                cbar_kws={"label": "$D_g$ (log2 mean 3' UTR length, fraction - Total)"})
    ax.set(xlabel="", ylabel="", title=f"40 genes called in most cohorts (* padj < 0.05)")
    ax.tick_params(axis="y", labelsize=6)
    color_ticklabels(ax.get_xticklabels(), [cohort_info.loc[c, "fraction"] for c in COHORTS])
    savefig(fig, "apa", "apa.top_genes.D_heatmap")
    plt.show()

In [ ]:
# APA shift and nuclear contamination: MALAT1 reads per read on nuclear protein-coding genes, relative to Total
# (as in ONT_organelles_vs_Total.ipynb), against the median D_g of the cohort
nuc_pc = genes_df.index[(genes_df["gene_type"] == "protein_coding") & (genes_df["chrom"] != "chrM")]
malat1 = genes_df.index[genes_df["gene_name"] == "MALAT1"]
gc_m, gc_n = gene_counts.reindex(malat1, fill_value=0), gene_counts.reindex(nuc_pc, fill_value=0)
tot_s = meta.loc[meta["fraction"] == "Total", "sample"]
R_malat1 = (gc_m.sum() / gc_n.sum()) / (gc_m[tot_s].sum().sum() / gc_n[tot_s].sum().sum())
R_cohort = np.exp(np.log(R_malat1.reindex(meta["sample"]).clip(lower=1e-2)).groupby(meta.set_index("sample")["cohort"]).mean())
apa_summary["R_MALAT1"] = R_cohort.reindex(apa_summary.index)
fig, ax = plt.subplots(figsize=(4.6, 3.8))
for c, r in apa_summary.iterrows():
    ax.scatter(np.log2(r["R_MALAT1"]), r["mean_log2FC_length"], color=FRACTION_COLORS[cohort_info.loc[c, "fraction"]],
               marker=BATCH_MARKERS[cohort_info.loc[c, "batch"]], s=50, edgecolor="black", lw=0.4)
rho, p = spearmanr(np.log2(apa_summary["R_MALAT1"]), apa_summary["mean_log2FC_length"])
ax.axhline(0, color="grey", lw=0.8)
ax.set(xlabel="nuclear contamination, log2 R MALAT1", ylabel="mean $D_g$ (3' UTR length,\nfraction - Total)",
       title=f"Spearman {rho:.2f} (p = {p:.2g}), n = {len(apa_summary)}")
ax.legend(handles=notation_handles(cohort_info["fraction"], cohort_info["batch"]), fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1))
sns.despine(); fig.tight_layout()
savefig(fig, "apa", "apa.mean_D_vs_nuclear_contamination")
plt.show()

### Interpretation of the cleavage-site usage and APA

- **Cleavage sites.** Without clustering there are 596,094 positions in 23,680 genes; the 163,256 with at least 5 reads carry 98.8% of the reads. 91% of the reads end at terminal positions, 6.6% at other exonic, 0.5% at intronic and 1.8% outside annotated genes. 99.4% of the terminal and 98.7% of the intronic positions lie within 10 nt of a PolyASite 3.0 cluster; exonic positions only 57%, because most of their reads are mitochondrial transcripts (no PolyASite clusters on chrM). The share of exonic reads follows the mitochondrial content of the library (Total 5%, ER 17%, PEX 18%, TJ 23%, of which 15–20 points chrM).
- **Read 3' ends scatter, but one position usually dominates.** Grouping the positions of a gene within 12 nt (figure only): 29,840 groups with ≥ 20 reads, median 9 positions per group, and the most used position carries a median 65% of the group's reads (≥ 50% in 73% of the groups). Keeping the positions separate therefore mostly measures the dominant site cleanly and leaves the minor ones near the prior, as intended.
- **CS usage differs from Total mainly in Golgi, PEX, Endo and TJ:** 1,746 / 4,713 / 2,449 / 11,993 positions (padj < 0.05) in 1,002 / 1,820 / 1,237 / 2,970 genes, against 53–410 in Free, Org2, ER and TIS of the three batches. The exception is the single-library ER/TIS Free (1,586, mostly terminal positions gained): with one library, the replicate spread is not estimated and the test is liberal. Significant changes are large (median |ΔΔ| 1.7–2.8 log2 odds); terminal positions change both ways (one tandem position gains what another loses). Intronic positions are mostly used **less** than in Total (TJ 428 lower / 58 higher, PEX 252 / 15, Endo 155 / 6, Golgi 112 / 6), as are other exonic positions; 74–94% of the exonic positions with lower usage in TJ, PEX, ER and TIS are on chrM, i.e. within-gene shifts among the 3' ends of mitochondrial transcripts.
- **3' UTRs (PAQR3 terminal exons, 7,512 genes) are longer in the same four fractions:** Golgi 122 genes lengthening / 14 shortening, PEX 193 / 50, Endo 121 / 25, TJ 236 / 102; the other cohorts have at most 19 calls, except TIS (42 shortening, 1 library). Fewer genes are called than with clustered CS (e.g. TJ 384 / 133 before), because the reads of a 3' end are now split over its positions and the per-gene estimate is more uncertain. The two-site test agrees: distal positions gain against the most proximal one (Golgi 42 up / 1 down, PEX 94 / 6, Endo 80 / 1, TJ 274 / 29), and $D_g$ correlates with it in every cohort (Spearman 0.25–0.36; lower than before, as the most proximal position is now often a minor one).
- **The shift is small and concerns a subset of genes.** The median $|D_g|$ is 0.005–0.007 log2 units (< 0.5%) in every cohort and the interquartile range stays within ±0.03; the mean $D_g$ is +0.014 to +0.023 in Golgi, PEX, Endo and TJ, about 0 in Free and Org2, and slightly negative in ER (−0.035) and TIS (−0.023): in ER, 21% of the genes have $D_g$ < −0.05 against 5% > +0.05, a trend towards shorter 3' UTRs that does not reach significance for individual genes (2 calls).
- **The APA shift tracks nuclear contamination across cohorts** (mean $D_g$ vs log2 $R$ of MALAT1: Spearman 0.77, p = 0.003, n = 12; net lengthening minus shortening calls: 0.96), but not the mitochondrial share of the reads (0.08). The median $D_g$ used before is ≈ 0 in every cohort with per-position CS, so the figure now shows the mean.
- **What it means is open.** As with clustered CS: the four fractions contain both more nuclear RNA and the membrane-associated material of their sorts, and 3' UTR-dependent membrane localization and nuclear retention of long-3' UTR isoforms would both lengthen 3' UTRs. The association with MALAT1 favours the nuclear explanation; a nuclear fraction prepared and sequenced with the sorts would test it directly. The shortening trend in ER and TIS, which are not nuclear-contaminated, is the candidate for a genuine ER-related signal.


## fracSanity on cleavage sites: $\alpha$ of every 3' end isoform

fracSanity per cohort on the CS counts (Sanity of Total and fractions separately, as for genes): $\alpha_j$ of every CS $j$, the fraction of the molecules ending at $j$ that are in the sorted fraction. For two CS of one gene, $\log_2\alpha_{j} - \log_2\alpha_{k}$ does not depend on $F$; it is the log2 ratio of their relative usage in the fraction and in Total. Below, for genes with at least two CS in the PAQR3 terminal exon: $\Delta = \log_2\alpha_{\text{distal}} - \log_2\alpha_{\text{proximal}}$ (longest vs shortest 3' UTR among the CS of the gene in the fit; posterior means), per cohort, compared with $D_g$ of the APA analysis.

In [ ]:
cs_alpha = []
for c, f in fits[CS_LEVEL].items():
    cs_alpha.append(pd.DataFrame({"cs_id": f["gene_ids"], "alpha": f["alpha_g"], "alpha_CI_lower": f["alpha_g_CI_lower"],
                                  "alpha_CI_upper": f["alpha_g_CI_upper"], "log2_alpha_mean": f["log2_alpha_mean"],
                                  "log2_alpha_sd": np.sqrt(f["log2_alpha_var"])}).assign(cohort=c))
cs_alpha = pd.concat(cs_alpha, ignore_index=True).merge(used[["gene_id", "gene_name", "site_class", "in_terminal_exon", "utr_length", "cs_pos"]], left_on="cs_id",
                                                        right_index=True, how="left")
save_tsv(cs_alpha, "alpha_cs", "fracSanity_alpha.cs.per_cohort.tsv.gz")

term = cs_alpha.loc[cs_alpha["in_terminal_exon"].fillna(False).astype(bool)]
grp = term.sort_values("utr_length").groupby(["cohort", "gene_id"])
prox, dist, n_sites = grp.first(), grp.last(), grp.size()     # shortest and longest 3' UTR of the gene
dd_cs = pd.DataFrame({"n_sites": n_sites, "delta_log2_alpha_distal_proximal": dist["log2_alpha_mean"] - prox["log2_alpha_mean"],
                      "log2_length_ratio": np.log2(dist["utr_length"] / prox["utr_length"])}).loc[n_sites >= 2].reset_index()
dd_cs = dd_cs.merge(apa_long[["gene_id", "cohort", "log2FC_length"]], on=["gene_id", "cohort"], how="left")
save_tsv(dd_cs, "alpha_cs", "delta_log2_alpha.distal_vs_proximal.per_gene.tsv.gz")
s = dd_cs.groupby("cohort").apply(lambda d: pd.Series({"genes": len(d), "median_delta": d["delta_log2_alpha_distal_proximal"].median(),
                                                       "share_positive": (d["delta_log2_alpha_distal_proximal"] > 0).mean(),
                                                       "spearman_vs_D_length": spearmanr(d["delta_log2_alpha_distal_proximal"], d["log2FC_length"],
                                                                                         nan_policy="omit")[0]})).reindex(COHORTS)
display(s.round(3))
save_tsv(s.reset_index(), "alpha_cs", "delta_log2_alpha.distal_vs_proximal.summary.tsv")

sns.set(font_scale=0.85, style="ticks")
fig, axes = plt.subplots(1, 2, figsize=(13, 4.0), gridspec_kw={"width_ratios": [1.6, 1]})
ax = axes[0]
sns.boxplot(data=dd_cs, x="cohort", y="delta_log2_alpha_distal_proximal", order=COHORTS, hue="cohort",
            palette={c: FRACTION_COLORS[cohort_info.loc[c, "fraction"]] for c in COHORTS}, showfliers=False, ax=ax, legend=False, width=0.6)
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(range(len(COHORTS)), COHORTS, rotation=90, fontsize=7)
color_ticklabels(ax.get_xticklabels(), [cohort_info.loc[c, "fraction"] for c in COHORTS])
ax.set(xlabel="", ylabel=r"$\log_2\alpha_{distal} - \log_2\alpha_{proximal}$")
ax = axes[1]
c0 = max(COHORTS, key=lambda c: (dd_cs["cohort"] == c).sum())
d = dd_cs.loc[dd_cs["cohort"] == c0]
ax.scatter(d["log2FC_length"], d["delta_log2_alpha_distal_proximal"], s=3, alpha=0.3, color=FRACTION_COLORS[cohort_info.loc[c0, "fraction"]], rasterized=True)
ax.axhline(0, color="grey", lw=0.6); ax.axvline(0, color="grey", lw=0.6)
ax.set(xlabel="$D_g$ (APA analysis)", ylabel=r"$\Delta\log_2\alpha$ distal - proximal", title=f"{c0}: Spearman {s.loc[c0, 'spearman_vs_D_length']:.2f}")
sns.despine(); fig.tight_layout()
savefig(fig, "alpha_cs", "delta_log2_alpha.distal_vs_proximal")
plt.show()

### Interpretation of the CS-level fracSanity

- **$F$ of the CS fits is not identified either** (156,937 positions in Total and fractions; posterior medians 0.0001–0.002, all profile likelihoods flat).
- **Within genes, the distal 3' end is slightly enriched in Golgi, PEX, Endo and TJ and depleted elsewhere:** median $\log_2\alpha_{distal} - \log_2\alpha_{proximal}$ = +0.001 to +0.014 with 50–51% of the 7,454 genes positive, against −0.012 to −0.055 (40–47% positive) in Free, Org2, ER and TIS; ER is the most negative (−0.055, 40% positive), in line with its shortening trend in the APA analysis.
- **The effect is weaker than with clustered CS** (+0.06 to +0.10 before): "distal" and "proximal" are now the extreme positions of the terminal exon, often minor positions whose $\alpha$ stays close to the prior. It agrees with the APA analysis (Spearman 0.28–0.39 with $D_g$ in every cohort), although the two use different Sanity runs (Total and fractions normalised separately vs jointly) and different models.


## Summary

- **Data.** 24 siCtrl fraction libraries (Free, Org2, ER, TIS, Golgi, PEX, Endo, TJ; three batches; 1–3 libraries per fraction and batch) against the average of three siCtrl Total libraries of HCT116; all genes of the annotation (21,848 with reads in both), 163,256 cleavage-site positions (not clustered; ≥ 5 reads); Sanity (EAP), fracSanity, ISMARA-style set activities and the isoform / APA tools of Sanity_utils.
- **$F$ cannot be determined without external information,** as in the P-body analysis; the ranking of genes within a fraction ($r_g$, $D_g$) can. The single-library ER/TIS Free gives an artefactual $F$ = 0.52.
- **Golgi, PEX, Endo and TJ form one group** in every analysis: PC1 of the expression, the number of over- and under-represented mRNAs, the gene-set profiles, the cleavage-site usage and APA. Free, Org2, ER and TIS are close to Total. This is the order of nuclear contamination (MALAT1) in `ONT_organelles_vs_Total.ipynb`.
- **Over- and under-represented mRNAs.** Small non-coding RNA hosts, lncRNAs and histone mRNAs are depleted from the sorted fractions; mRNAs of ER-translated proteins are depleted from Free and enriched in the membrane sorts (Golgi, PEX, Endo, TJ), ribosomal-protein mRNAs the opposite: a membrane-bound vs free-cytosolic signature, beyond mRNA abundance. Markers of the individual organelles are not enriched in their own sorts.
- **APA (tandem, PAQR3 terminal exons).** Golgi, PEX, Endo and TJ have longer 3' UTRs than Total in 121–236 genes (shortening in 14–102), and intronic 3' ends are used less there; the shift is small (median $|D_g|$ < 0.5%, mean +0.014 to +0.023 log2) and is seen in the CS-level fracSanity too (distal positions slightly enriched within genes). Free and Org2 show almost no APA difference; ER and TIS lean towards shorter 3' UTRs (not significant per gene). The lengthening follows nuclear contamination across cohorts (mean $D_g$ vs MALAT1: Spearman 0.77), not the mitochondrial content. Intronic polyadenylation as such is not analysed yet (isoform assignment of `nanoflowz` to be checked).
- **Caveats.** The Total reference was prepared and sequenced separately from the fractions (batch); the fraction libraries are shallow (0.1–3.1 M reads on the well-expressed nuclear protein-coding genes, against 13–17 M in Total), several ER/TIS libraries were excluded for depth, and ER/TIS Free, Org2 and TIS have a single library each; the Golgi, PEX, Endo and TJ effects cannot yet be separated from their nuclear contamination.
- **Next steps:** a nuclear fraction and a Total library prepared and sequenced with the sorts; spike-ins (or sorted-event counts and RNA yield) to determine $F$; siCPSF5 vs siCtrl once the siCtrl patterns are settled.